In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [6]:
# Create directories
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('models', exist_ok=True)

print("--- Project Directory Structure Initialized ---")

# Ingest initial raw financial and sporting data
raw_club_data = {
    'Club_ID': [
        'REAL_MADRID', 'MAN_CITY', 'FC_BARCELONA', 'MAN_UTD', 
        'BAYERN_MUNICH', 'PSG', 'LIVERPOOL', 'ARSENAL', 'TOTTENHAM', 'CHELSEA'
    ],
    'League': [
        'La Liga', 'Premier League', 'La Liga', 'Premier League',
        'Bundesliga', 'Ligue 1', 'Premier League', 'Premier League', 
        'Premier League', 'Premier League'
    ],
    'Matchday_Revenue_M': [122.0, 85.0, 145.0, 130.0, 110.0, 150.0, 115.0, 125.0, 120.0, 95.0],
    'Broadcast_Revenue_M': [310.0, 340.0, 270.0, 290.0, 220.0, 190.0, 300.0, 280.0, 250.0, 260.0],
    'Commercial_Revenue_M': [399.0, 401.0, 385.0, 330.0, 414.0, 461.0, 268.0, 127.0, 179.0, 157.0],
    'Total_Wage_Bill_M': [452.0, 510.0, 528.0, 385.0, 325.0, 626.0, 373.0, 269.0, 251.0, 404.0],
    'Squad_Market_Value_M': [1340.0, 1270.0, 940.0, 870.0, 930.0, 900.0, 920.0, 1110.0, 780.0, 950.0],
    'Net_Debt_M': [40.0, 300.0, 600.0, 750.0, 0.0, 150.0, 80.0, 600.0, 850.0, 1000.0],
    'UEFA_Coefficient_Rank': [1, 2, 6, 9, 3, 7, 5, 12, 17, 10],
    'Enterprise_Value_M': [6000, 5500, 4800, 5000, 4500, 4200, 4300, 3100, 3000, 3200]
}

df_raw = pd.DataFrame(raw_club_data)
df_raw['Total_Revenue_M'] = (
    df_raw['Matchday_Revenue_M'] + 
    df_raw['Broadcast_Revenue_M'] + 
    df_raw['Commercial_Revenue_M']
)

output_path = 'data/raw/clubs_financials_raw.csv'
df_raw.to_csv(output_path, index=False)

print(f"--- Raw Dataset Successfully Saved to {output_path} ---")
print(df_raw[['Club_ID', 'Total_Revenue_M', 'Total_Wage_Bill_M', 'Enterprise_Value_M']].head())

--- Project Directory Structure Initialized ---
--- Raw Dataset Successfully Saved to data/raw/clubs_financials_raw.csv ---
         Club_ID  Total_Revenue_M  Total_Wage_Bill_M  Enterprise_Value_M
0    REAL_MADRID            831.0              452.0                6000
1       MAN_CITY            826.0              510.0                5500
2   FC_BARCELONA            800.0              528.0                4800
3        MAN_UTD            750.0              385.0                5000
4  BAYERN_MUNICH            744.0              325.0                4500


In [7]:
# 1. Load the raw data we just saved in Phase 1
df = pd.read_csv('data/raw/clubs_financials_raw.csv')

print("--- Loaded Raw Data for Feature Engineering ---")

# 2. Compute FFP Compliance Ratios (UEFA guidelines suggest wage-to-revenue <= 70%)
df['Wage_to_Revenue_Ratio'] = df['Total_Wage_Bill_M'] / df['Total_Revenue_M']
df['Net_Debt_to_Revenue'] = df['Net_Debt_M'] / df['Total_Revenue_M']

# 3. Compute Asset & Efficiency Multiples
df['Squad_Value_to_Revenue'] = df['Squad_Market_Value_M'] / df['Total_Revenue_M']

# 4. Create an FFP Compliance Flag (True if wage burden is safe, False if at risk of breaking rules)
df['FFP_Compliant'] = df['Wage_to_Revenue_Ratio'] <= 0.70

# 5. Save the enriched data to your processed folder as specified in your plan
processed_path = 'data/processed/master_club_analytics.csv'
df.to_csv(processed_path, index=False)

print(f"--- Feature Engineering Complete! Saved to {processed_path} ---")
print("\nPreview of Financial Ratios:")
print(df[['Club_ID', 'Wage_to_Revenue_Ratio', 'FFP_Compliant', 'Squad_Value_to_Revenue']].head())

--- Loaded Raw Data for Feature Engineering ---
--- Feature Engineering Complete! Saved to data/processed/master_club_analytics.csv ---

Preview of Financial Ratios:
         Club_ID  Wage_to_Revenue_Ratio  FFP_Compliant  Squad_Value_to_Revenue
0    REAL_MADRID               0.543923           True                1.612515
1       MAN_CITY               0.617433           True                1.537530
2   FC_BARCELONA               0.660000           True                1.175000
3        MAN_UTD               0.513333           True                1.160000
4  BAYERN_MUNICH               0.436828           True                1.250000


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Load your processed master dataset
df = pd.read_csv('data/processed/master_club_analytics.csv')

# 2. Define Features (X) and Target (y)
# We use financial streams, wage burden, asset value, debt, and prestige (UEFA rank) to predict Enterprise Value
features = [
    'Total_Revenue_M', 
    'Total_Wage_Bill_M', 
    'Squad_Market_Value_M', 
    'Net_Debt_M', 
    'UEFA_Coefficient_Rank',
    'Wage_to_Revenue_Ratio'
]

X = df[features]
y = df['Enterprise_Value_M']

# 3. Train / Test Split (80% training data, 20% testing data)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Initialize and train a Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 5. Make predictions and evaluate performance
predictions = model.predict(X_test)

print("--- Machine Learning Valuation Results ---")
print(f"Mean Absolute Error: €{mean_absolute_error(y_test, predictions):.2f}M")
print(f"Model R² Score: {r2_score(y_test, predictions):.2f}\n")

# 6. Feature Importance Analysis (What drives club value the most?)
importances = pd.DataFrame({
    'Financial Feature': features,
    'Importance Score': model.feature_importances_
}).sort_values(by='Importance Score', ascending=False)

print("--- Key Valuation Drivers (Feature Importance) ---")
print(importances)

--- Machine Learning Valuation Results ---
Mean Absolute Error: €461.00M
Model R² Score: 0.83

--- Key Valuation Drivers (Feature Importance) ---
       Financial Feature  Importance Score
0        Total_Revenue_M          0.336376
4  UEFA_Coefficient_Rank          0.283458
2   Squad_Market_Value_M          0.171960
5  Wage_to_Revenue_Ratio          0.080400
3             Net_Debt_M          0.072304
1      Total_Wage_Bill_M          0.055501
